# <font color = 'red'> DEPENDENCIAS

In [26]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [27]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [28]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [29]:
col = "Monthly_Inhand_Salary"

## <font color = 'skyblue'> ANÁLISIS GENERAL

La mediana de los ingresos de los clientes malos parecen significativamente inferiores a los clientes Standard y en mayor medida a los Buenos.

In [30]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [31]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [32]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Monthly_Inhand_Salary_Decile,,,,,,,,,,
0,303.645417,1105.052500,10003,0.10003,4710,579,4714,0.470859,0.057883,0.471259
1,1105.126667,1469.108333,9999,0.09999,3544,1946,4509,0.354435,0.194619,0.450945
2,1469.135000,1826.489167,10005,0.10005,3396,2198,4411,0.339430,0.219690,0.440880
3,1828.206667,2564.049167,9997,0.09997,1324,2693,5980,0.132440,0.269381,0.598179
4,2564.368333,3096.378333,9999,0.09999,2009,3483,4507,0.200920,0.348335,0.450745
5,3096.836667,4044.262500,10000,0.10000,2350,4137,3513,0.235000,0.413700,0.351300
6,4045.392500,5308.210000,10000,0.10000,3576,1549,4875,0.357600,0.154900,0.487500
7,5309.226667,6712.043333,9998,0.09998,2334,2728,4936,0.233447,0.272855,0.493699
8,6718.416667,9121.490000,9999,0.09999,525,4082,5392,0.052505,0.408241,0.539254


No missing values found.
No infinite values found.
No duplicate rows found.


In [33]:
df[(df[continuous_variable] >= 5309.226667) & (df[continuous_variable] <= 6712.043333) & (df['Credit_Score'] == 2)].shape

(2728, 85)

Porporción de Buenos: aunque se observa una relación positiva entre los ingresos netos y la proporción de buenos, se observa un
cambio abrupto entre los deciles 6 y 7, al pasar de una proporción de buenos de 41% a 15%,
lo cual no es razonable.

Proporción de standard: aunque sí existe una tendencia negativa entre la proporción de Standard y los ingresos
hay cambios irregulares que son poco razonables.

Proporción de malos: de manera similar, la proporción de malos y los ingresos tienen una relación inversa, sin embargo, 
hay cambios irregulares abruptos.

In [34]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y los ingresos:

In [35]:
group_map = {0: "Group_1", 
             1: "Group_2", 
             2: "Group_3", 
             3: "Group_4", 
             4: "Group_4",
             5: "Group_4", 
             6: "Group_4", 
             7: "Group_4", 
             8: "Group_5", 
             9: "Group_5"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3,
    'Group_4': 4,
    'Group_5': 5
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Monthly_Inhand_Salary,,,,,,,,,,
Group_1,303.645417,1105.052500,10003,0.10003,4710,579,4714,0.470859,0.057883,0.471259
Group_2,1105.126667,1469.108333,9999,0.09999,3544,1946,4509,0.354435,0.194619,0.450945
Group_3,1469.135000,1826.489167,10005,0.10005,3396,2198,4411,0.339430,0.219690,0.440880
Group_4,1828.206667,6712.043333,49994,0.49994,11593,14590,23811,0.231888,0.291835,0.476277
Group_5,6718.416667,15204.633333,19999,0.19999,525,11071,8403,0.026251,0.553578,0.420171


No missing values found.
No infinite values found.
No duplicate rows found.


In [36]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [37]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [38]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Monthly_Inhand_Salary,"100,000.00","4,198.77","3,187.49",303.65,"1,626.76","3,096.38","5,961.74","15,204.63"


Todos los coeficientes son significativos.

Monthly_Inhand_Salary_Scaled 3.2222: según lo esperado, el coeficiente es positivo: por cada dólar adicional de ingreso neto, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.4315: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7813: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [39]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


Optimization terminated successfully.
         Current function value: 0.999169
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -99917.
Model:                   OrderedModel   AIC:                         1.998e+05
Method:            Maximum Likelihood   BIC:                         1.999e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:20:30                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Monthly_Inhand_Salary_Decile 0.2162: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.2910: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7741: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [40]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

Optimization terminated successfully.
         Current function value: 1.008567
         Iterations: 10
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0086e+05
Model:                   OrderedModel   AIC:                         2.017e+05
Method:            Maximum Likelihood   BIC:                         2.017e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:20:31                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Monthly_Inhand_Salary 0.5537: Por cada grupo adcional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 0.7172: Umbral que separa las categorías Bad y Standard.

Threshold 1/2	0.7862: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [41]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

Optimization terminated successfully.
         Current function value: 1.001425
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0014e+05
Model:                   OrderedModel   AIC:                         2.003e+05
Method:            Maximum Likelihood   BIC:                         2.003e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:20:33                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Monthly_Inhand_Salary` usando Regresión Ordinal

| Representación                         | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Monthly_Inhand_Salary_Scaled`        | 3.2222                | **Log-Likelihood**: -99,917<br>**AIC**: 199,834<br>**BIC**: 199,875                    | 🔹 Modelo con mejor ajuste global.<br>🔹 Muy informativa, conserva el detalle continuo sin problemas de escala. |
| `Monthly_Inhand_Salary_Decile`        | 0.2162                | **Log-Likelihood**: -100,856<br>**AIC**: 201,713<br>**BIC**: 201,755                   | 🔹 Peor ajuste comparado con las otras opciones.<br>🔹 La discretización en deciles reduce la información. |
| `Grouped_Monthly_Inhand_Salary`       | 0.5537                | **Log-Likelihood**: -100,142<br>**AIC**: 200,285<br>**BIC**: 200,326                   | 🔹 Ajuste intermedio.<br>🔹 Agrupar deciles mejora la interpretabilidad sin afectar tanto el desempeño. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
